optuna
- optuna is a hyperparameter optimization framework that automates the process of finding the best hyperparameters for machine learning models. It uses techniques like Bayesian optimization, random search, and grid search to efficiently explore the hyperparameter space and identify the optimal configuration for a given model and dataset. Optuna allows users to define an objective function that evaluates the performance of a model based on its hyperparameters, and it iteratively suggests new hyperparameter values to improve the model's performance.

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
from torchinfo import summary

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder

import pandas as pd


In [124]:
df = pd.read_csv('https://raw.githubusercontent.com/JosePadillaMtnz/BreastCancer_KaggleDataset/main/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [125]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                569 non-null    str    
 1   radius_mean              569 non-null    float64
 2   texture_mean             569 non-null    float64
 3   perimeter_mean           569 non-null    float64
 4   area_mean                569 non-null    float64
 5   smoothness_mean          569 non-null    float64
 6   compactness_mean         569 non-null    float64
 7   concavity_mean           569 non-null    float64
 8   concave points_mean      569 non-null    float64
 9   symmetry_mean            569 non-null    float64
 10  fractal_dimension_mean   569 non-null    float64
 11  radius_se                569 non-null    float64
 12  texture_se               569 non-null    float64
 13  perimeter_se             569 non-null    float64
 14  area_se                  569 non-null

In [126]:
X = df.drop(columns=['diagnosis']).values
y = df['diagnosis'].values

# X = df.drop(columns=['diagnosis']).values
# X.shape

In [127]:
X.shape, y.shape

((569, 30), (569,))

In [128]:
train_X, test_X, train_y, test_y = train_test_split(X,y,test_size=0.2, random_state=42)

scaler = StandardScaler()
train_X = scaler.fit_transform(train_X)
test_X = scaler.transform(test_X)


label_encoder = LabelEncoder()
train_y = label_encoder.fit_transform(train_y)
test_y = label_encoder.transform(test_y)


train_X = torch.from_numpy(train_X).float()
train_y = torch.from_numpy(train_y).float()
test_X = torch.from_numpy(test_X).float()
test_y = torch.from_numpy(test_y).float()

In [129]:
train_X.shape

torch.Size([455, 30])

In [130]:
# create custom dataset
class BreastCancerDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, idx):
    return self.features[idx], self.labels[idx]

In [131]:
# create model
class BreastCancer_model(nn.Module):
  def __init__(self, input_dim, output_dim,num_hidden_units, num_hidden_layers,dropout_rate):
    super().__init__()


    layers = []
    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_dim, num_hidden_units))
      # Add batch normalization layer after linear layer to stabilize and accelerate training
      # Batch normalization normalizes the output of the previous layer by subtracting the batch mean and 
      # dividing by the batch standard deviation, which helps to reduce internal covariate shift and allows for faster convergence.
      
      # internal covariate shift refers to the change in the distribution of network activations due to changes in 
      # the parameters of the previous layers during training, which can slow down training and make it more difficult 
      # for the model to converge.
      layers.append(nn.BatchNorm1d(num_hidden_units))
      
      layers.append(nn.ReLU())
      
      # Add dropout layer after activation function to prevent overfitting 
      # Dropout randomly sets a fraction of the input units to 0 at each update during training time, 
      # which helps prevent overfitting by adding noise to the model and forcing it to learn more robust features.
      layers.append(nn.Dropout(dropout_rate))
    
      input_dim = num_hidden_units


    
    layers.append(nn.Linear(num_hidden_units, output_dim))
    layers.append(nn.Sigmoid())

    # * operator is used to unpack the list of layers and pass them as individual arguments to nn.Sequential
    self.model = nn.Sequential(*layers)



 
  def forward(self, x):
    return self.model(x)
   

In [132]:
train_dataset = BreastCancerDataset(train_X, train_y)
test_dataset = BreastCancerDataset(test_X, test_y)

train_dataset_loader = DataLoader(train_dataset, batch_size=100, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=100)

In [133]:
learning_rate = 0.01
epochs = 30
input_dim = X.shape[1]
output_dim = 1
num_hidden_units = 12
num_hidden_layers = 2
dropout_rate = 0.2

In [134]:
model = BreastCancer_model(input_dim, output_dim,num_hidden_units, num_hidden_layers,dropout_rate)
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [135]:
summary(model, input_size=(1, X.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
BreastCancer_model                       [1, 1]                    --
├─Sequential: 1-1                        [1, 1]                    --
│    └─Linear: 2-1                       [1, 12]                   372
│    └─BatchNorm1d: 2-2                  [1, 12]                   24
│    └─ReLU: 2-3                         [1, 12]                   --
│    └─Dropout: 2-4                      [1, 12]                   --
│    └─Linear: 2-5                       [1, 12]                   156
│    └─BatchNorm1d: 2-6                  [1, 12]                   24
│    └─ReLU: 2-7                         [1, 12]                   --
│    └─Dropout: 2-8                      [1, 12]                   --
│    └─Linear: 2-9                       [1, 1]                    13
│    └─Sigmoid: 2-10                     [1, 1]                    --
Total params: 589
Trainable params: 589
Non-trainable params: 0
Total mult-adds (Un

In [136]:
for epoch in range(epochs):
    epoch_loss = 0.0
    
    for features, labels in train_dataset_loader:
        optimizer.zero_grad()
        
        outputs = model(features)
        loss = loss_fn(outputs, labels.view(-1, 1))
        
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()

    print(f"Epoch: {epoch+1}/{epochs}, Loss: {epoch_loss / len(train_dataset_loader):.4f}")

Epoch: 1/30, Loss: 0.6792
Epoch: 2/30, Loss: 0.6605
Epoch: 3/30, Loss: 0.6510
Epoch: 4/30, Loss: 0.6444
Epoch: 5/30, Loss: 0.6267
Epoch: 6/30, Loss: 0.6332
Epoch: 7/30, Loss: 0.6156
Epoch: 8/30, Loss: 0.6026
Epoch: 9/30, Loss: 0.5921
Epoch: 10/30, Loss: 0.5888
Epoch: 11/30, Loss: 0.5783
Epoch: 12/30, Loss: 0.5736
Epoch: 13/30, Loss: 0.5715
Epoch: 14/30, Loss: 0.5552
Epoch: 15/30, Loss: 0.5479
Epoch: 16/30, Loss: 0.5561
Epoch: 17/30, Loss: 0.5317
Epoch: 18/30, Loss: 0.5261
Epoch: 19/30, Loss: 0.5209
Epoch: 20/30, Loss: 0.5067
Epoch: 21/30, Loss: 0.5196
Epoch: 22/30, Loss: 0.4960
Epoch: 23/30, Loss: 0.5033
Epoch: 24/30, Loss: 0.4955
Epoch: 25/30, Loss: 0.4883
Epoch: 26/30, Loss: 0.4892
Epoch: 27/30, Loss: 0.4794
Epoch: 28/30, Loss: 0.4681
Epoch: 29/30, Loss: 0.4618
Epoch: 30/30, Loss: 0.4586


In [137]:
model.eval()
accuracy_list = []
with torch.no_grad():
  for batch_feature, batch_label in  test_dataloader:
    y_pred = model(batch_feature)
    y_pred = (y_pred > 0.8).float()
    accuracy = (y_pred.view(-1) == batch_label).float().mean().item()
    accuracy_list.append(accuracy)


sum(accuracy_list)/len(accuracy_list)

0.7171428799629211

In [140]:
# from torchmetrics.classification import BinaryAccuracy

# metric = BinaryAccuracy(threshold=0.8)

# model.eval()

# with torch.no_grad():
#     for X_batch, y_batch in test_dataloader:
#         outputs = model(X_batch)
#         metric.update(outputs.view(-1), y_batch)

# accuracy = metric.compute()
# print(accuracy)

In [141]:
import optuna

/Users/shubham/Desktop/pytorch/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [146]:
def objective(trial):
    # Define the hyperparameters to optimize
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    num_hidden_units = trial.suggest_int('num_hidden_units', 8, 128)
    num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
    dropout_rate = trial.suggest_float('dropout_rate', 0.0, 0.5)
    batch_size = trial.suggest_int('batch_size', 16, 128)

    train_dataset_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

    # Create the model with the suggested hyperparameters
    model = BreastCancer_model(input_dim, output_dim, num_hidden_units, num_hidden_layers, dropout_rate)
    loss_fn = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    # Train the model for a few epochs
    for epoch in range(epochs):
        epoch_loss = 0.0
        
        for features, labels in train_dataset_loader:
            optimizer.zero_grad()
            
            outputs = model(features)
            loss = loss_fn(outputs, labels.view(-1, 1))
            
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()

    # Evaluate the model on the test set
    metric = BinaryAccuracy(threshold=0.8)
    model.eval()
    
    with torch.no_grad():
        for X_batch, y_batch in test_dataloader:
            outputs = model(X_batch)
            metric.update(outputs.view(-1), y_batch)

    accuracy = metric.compute().item()
    
    return accuracy

In [147]:
# Create an Optuna study and optimize the objective function
# direction='maximize' tells Optuna to maximize the objective function (accuracy in this case) during the optimization process.
# The n_trials parameter specifies the number of optimization trials to perform, which determines how many different 
# combinations of hyperparameters will be evaluated.
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

[I 2026-03-22 00:28:03,155] A new study created in memory with name: no-name-c62b1caf-8895-4394-bbaa-ff2c855900c1
[I 2026-03-22 00:28:03,402] Trial 0 finished with value: 0.6315789222717285 and parameters: {'learning_rate': 0.00014550538611630826, 'num_hidden_units': 113, 'num_hidden_layers': 1, 'dropout_rate': 0.010685408104766925, 'batch_size': 68}. Best is trial 0 with value: 0.6315789222717285.
[I 2026-03-22 00:28:03,535] Trial 1 finished with value: 0.6228070259094238 and parameters: {'learning_rate': 0.0005513079686310029, 'num_hidden_units': 31, 'num_hidden_layers': 2, 'dropout_rate': 0.1695421110274301, 'batch_size': 103}. Best is trial 0 with value: 0.6315789222717285.
[I 2026-03-22 00:28:03,923] Trial 2 finished with value: 0.9649122953414917 and parameters: {'learning_rate': 0.04043184874548025, 'num_hidden_units': 125, 'num_hidden_layers': 4, 'dropout_rate': 0.25928083085914505, 'batch_size': 54}. Best is trial 2 with value: 0.9649122953414917.
[I 2026-03-22 00:28:04,008] T

In [148]:
study.best_params

{'learning_rate': 0.04043184874548025,
 'num_hidden_units': 125,
 'num_hidden_layers': 4,
 'dropout_rate': 0.25928083085914505,
 'batch_size': 54}

In [149]:
study.best_value

0.9649122953414917